# Spine innervation analyses #1

In this notebook we select and download an `EM synaptome`. That is:
  - a skeleton representation of a single neuron from an electron microscopic dense tissue reconstruction
  - plus the spines on its surface, individually extracted
  - plus all afferent synapses onto the neuron, mapped to individual spines, shafts or the soma

Then we do some basic analyses of the innervation of spines by synapses:
  - Which types innervate the neuron where?
  - How many synapses innervate a spine?
  - Which types of synapses share a spine?

This notebook serves as a starting point to a new user. It shows ways to access synapse and spine information. 

## Platform authentication
We begin by authenticating with the OBI platform to be able to access the synaptomes.

Please follow the instruction below to authenticate, then select the project to work with. 
The synaptome you want to visualize must be either public, or generated within that project.


In [ ]:
import h5py
#import neurom
import bluepysnap as snap
#import pandas
import numpy

from pathlib import Path

from obi_notebook import get_environment
import obi_auth
from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities
from entitysdk import Client
from entitysdk.models import Circuit
from entitysdk.staging import stage_circuit

from analyses_for_spiny_morphs import with_binned_column
from analyses_for_spiny_synaptomes import get_synaptome_morphology, get_synaptome_synapses, spine_innervation_df, get_synaptome_innervating_neurons

environment = get_environment.get_environment()
token = obi_auth.get_token(environment=environment, auth_mode="daf")
project_context = get_projects(token)



## Selecting a synaptome

**IMPORTANT. Read this carefully** 

`Synaptomes` is what we call simulatable models of a neuron and its afferents. There are many on the OBI platform, but not all of them have been generated from electron microscopy. Others have been built as statistical models by stochastic algorithms.

For this notebook, we require a `Synaptome` from electron microscopy. Below, you will be provided with a table to select a `Synaptome` from. Please make sure you select one where the value in the last column is `em_reconstruction`. Otherwise, in the next cell an error will be raised.


In [ ]:
client = Client(environment=environment, token_manager=token, project_context=project_context)
circ_ids = []
circ_ids = get_entities(entity_type="circuit", token=token, result=circ_ids, project_context=project_context,
                        exclude_scales=["pair", "small", "microcircuit", "region", "system", "whole_brain"],
                        add_columns=["build_category"], page_size=50)

## Download the `Synaptome`. Load neuron morphology and synapses


In [ ]:
circ_entity = client.get_entity(entity_id=circ_ids[0], entity_type=Circuit)
if circ_entity.build_category != "em_reconstruction":
    raise ValueError("The selected synaptome is not derived from an electron microscopic reconstruction!")

# This downloads the selected Synaptome to the local system.
fn_synaptome = stage_circuit(client=client, model=circ_entity, output_dir=Path("downloaded_synaptome"))

# Load as a circuit
synaptome = snap.Circuit(fn_synaptome)

synapses = get_synaptome_synapses(synaptome)
spiny_morph = get_synaptome_morphology(synaptome)
afferents = get_synaptome_innervating_neurons(synaptome)

# Analyses

Now we begin with some analyses.

Here we ask: How many synapses are placed on the same spine? We provide a pie chart of the number of synapses per spine in the downloaded data.

We then plot how this depends on the length of a spine (longer spines are more likely to be multi-innervated) and how it depends on the distance of the spine from the soma.

These analyses teach you how to access synapse info, spines and how to calculate path distances.

In [ ]:
from matplotlib import pyplot as plt

spine_stats = spine_innervation_df(synapses=synapses, spiny_morph=spiny_morph)

pd_bins = numpy.arange(0, 450 + 25.0, 25.0)
pd_stats = with_binned_column(spine_stats, "soma_path_distance", pd_bins).groupby(
           ["soma_path_distance_binned", "count"])["count"].count().unstack("count", fill_value=0)
pd_probs = (pd_stats.transpose() / pd_stats.sum(axis=1)).transpose()

fig = plt.figure(figsize=(6, 5.5))
ax = fig.add_subplot(2, 2, 1)
count_incidence = spine_stats["count"].value_counts().sort_index()
ax.pie(count_incidence, labels=count_incidence.index.to_numpy(dtype=int))

ax = fig.add_subplot(2, 2, 2)

c_vs_l = spine_stats.set_index("count")["spine_length"]
len_bins = numpy.linspace(0, 4.0, 25)
len_bin_c = 0.5 * (len_bins[:-1] + len_bins[1:])

for c in range(4):
    h = numpy.histogram(c_vs_l[c], bins=len_bins, density=True)[0]
    ax.plot(len_bin_c, h, label=f"{c} syns")
ax.set_xlabel("Spine length (um)")
ax.set_ylabel("Prob. density")
plt.legend()

ax = fig.add_subplot(2, 2, 3)
c_vs_pd = spine_stats.set_index("count")["soma_path_distance"]
pd_bin_c = 0.5 * (pd_bins[:-1] + pd_bins[1:])

for c in range(4):
    h = numpy.histogram(c_vs_pd[c], bins=pd_bins, density=True)[0]
    ax.plot(pd_bin_c, h, label=f"{c} syns")
ax.set_xlabel("Spine soma path distance (um)")
ax.set_ylabel("Prob. density")
plt.legend()

ax = fig.add_subplot(2, 2, 4)
for c in range(4):
    ax.plot(pd_probs[c], label=f"{c} syns")
ax.set_xlabel("Spine soma path distance (um)")
ax.set_ylabel("P")

# What types of neurons share a spine?

Unfortunately, in EM datasets most synapses are from extrinsic neurons. 
Hence, this will probably not say much.

In [ ]:
src_id_per_spine = synapses.groupby("spine_sharing_id")["@source_node"].apply(list).drop(-1)
src_ids_two_syns = src_id_per_spine[src_id_per_spine.apply(len) == 2]

two_syn_types = src_ids_two_syns.apply(lambda _lst: afferents.loc[_lst, "synapse_class"].sort_values().reset_index(drop=True))
two_syn_types.value_counts().unstack(1, fill_value=0)

# Path distance of innervations

We ask: Which neurons innervate neurons close to the soma, which ones more distally?

Note that we analyze this for only the single post-synaptic neuron that we downloaded, but for all its innervating neurons!

The plot below shows the mean path distance to the soma from synapses of the indicated neuron types.

In [ ]:
from conntility.subcellular import MorphologyPathDistanceCalculator
from analyses_for_spiny_morphs import SOMA_SKELETON_LOC as soma_skeleton_loc

calc = MorphologyPathDistanceCalculator(spiny_morph.to_morphio())

type_and_pd = afferents.loc[synapses["@source_node"], ["cell_type"]]
type_and_pd["soma_path_distance"] = calc.path_distances(soma_skeleton_loc, synapses)[0]

mean_pds = type_and_pd.groupby("cell_type")["soma_path_distance"].mean()

plt.bar(range(len(mean_pds)), mean_pds)
_ = plt.gca().set_xticks(range(len(mean_pds)))
_ = plt.gca().set_xticklabels(mean_pds.index, rotation="vertical")
_ = plt.gca().set_ylabel("Mean soma path dist. (um)")
_ = plt.gca().set_xlabel("Innervating type")

# Which types innervate the soma?

In [ ]:
idx_on_soma = synapses.loc[synapses["afferent_section_id"] == 0, "@source_node"]
types_on_soma = afferents.loc[idx_on_soma.to_numpy(), "cell_type"].value_counts()
types_on_soma = types_on_soma[types_on_soma > 0]
_ = plt.pie(types_on_soma, labels=types_on_soma.index)

# Plot neuron with afferent synapses

By default the plot below shows only inervation from intrinsic synapses. Set `show_extrinsic_innervation` to True to show **all**.

In [ ]:
from neurom.view import plot_morph

show_extrinsic_innervation = False

plot_morph(spiny_morph, ax=plt.figure(figsize=(9, 9)).gca())

for pre_type in type_and_pd["cell_type"].drop_duplicates():
    if pre_type != "extrinsic_neuron" or show_extrinsic_innervation:
        xy = synapses.loc[(type_and_pd["cell_type"] == pre_type).to_numpy(),
                          ["afferent_synapse_x", "afferent_synapse_y"]]
        plt.scatter(xy["afferent_synapse_x"], 
                    xy["afferent_synapse_y"], s=int(2 + 5 * (pre_type == "PTC")),
                    label=pre_type)
        
plt.axis("equal")
plt.gca().set_ylabel("Depth (um)")
plt.legend()